# 선수별 3-fold에서 원판 검출 65·95에폭 비교

같은 선수를 두 번 비교합니다. 그 선수를 뺀 데이터로 YOLO26n을 한 번 학습하고, 학습 도중 저장한 65번째와 95번째 가중치를 같은 영상에 적용합니다. 원래 검증 선수 6명은 이번 비교에서 제외합니다.

VS Code에서 개인 프로젝트의 `.venv` 커널을 선택하고 위에서 아래로 실행하세요. 처음에는 1·2단계로 대상과 라벨을 확인하고, 3단계에서 학습을 켜세요.

## 1. 설정과 수동 허브 판독 확인

허브는 찍어 둔 원판 중심입니다. 직접 확인과 판독 불가를 구분하고, 아직 저장하지 않은 장면이 있는지 먼저 확인합니다.

In [ ]:
from pathlib import Path
import shutil

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

from wlcv.annotation.fold_hubs import fold_targets, progress_table
from wlcv.evaluation.fold_epochs import evaluate_fold, fold_truth, summarize_folds
from wlcv.training.plate_folds import OUTPUT, prepare_fold, run_dir, train_fold
from wlcv.training.plate_schedule import baseline_settings

FOLDS = [1, 2, 3]  # 한 회차만 실행할 때는 예를 들어 [1]로 바꾸세요.
if not FOLDS or len(set(FOLDS)) != len(FOLDS) or not set(FOLDS) <= {1, 2, 3}:
    raise ValueError("1, 2, 3 중 실행할 fold를 중복 없이 선택하세요.")
settings = baseline_settings()
keys = ["model", "device", "epochs", "patience", "imgsz", "batch", "seed", "optimizer"]
display(pd.Series({key: settings[key] for key in keys}, name="설정").to_frame())
counts = progress_table(fold_targets())
display(counts.groupby(level="Fold").sum(), counts)
if counts["미판독"].sum():
    raise ValueError("17번 허브 판독 노트북에서 미판독 프레임을 먼저 저장하세요.")
print(f"실행 폴더의 여유 공간 {shutil.disk_usage(run_dir(1).parent).free / 2**30:.1f} GiB")

## 2. 학습·검증 분할 확인

이 단계에서는 프레임을 준비합니다. 각 선수는 한 회차에서만 검증 대상입니다. 빈 라벨은 원판 없는 프레임이고, 허브 좌표는 평가에만 사용합니다. ‘동작 중’은 1차 ready부터 마지막으로 판독된 이벤트까지입니다. c05는 finish만 기록되어 있어 전체 양성 비교에 포함합니다.

In [ ]:
inputs = {}
for fold in FOLDS:
    data, split_csv = prepare_fold(fold, settings)
    inputs[fold] = (data, split_csv)
    split = pd.read_csv(split_csv)
    truth = fold_truth(data)
    n_train = len(list((data.parent / "labels/train").glob("*.txt")))
    print(f"Fold {fold} | 학습 {n_train}장 · 검증 {len(truth)}장 · 검증 양성 {truth.x1.count()}장 · 동작 중 {truth.in_motion.sum()}장")
    display(split.groupby("role").person_id.agg(", ".join).to_frame("선수"))

## 3. Mac에서 학습하기

아래 `START_TRAINING`을 `True`로 바꾸고 셀을 실행하세요. 회차는 차례대로 실행합니다. 각 회차는 원본 `yolo26n.pt`에서 출발해 100에폭을 학습하고 같은 실행의 65·95에폭을 복사합니다. 기존 베이스라인의 배치·시드·증강·학습률 일정을 그대로 사용합니다.

중단 후에는 같은 회차를 선택하고 1단계부터 다시 실행하세요. 완료된 회차는 저장 파일을 확인하고, 진행 중인 회차는 저장된 마지막 에폭부터 재개합니다. Mac 저장 공간이 부족하면 학습 전에 여유를 확보하세요.

In [ ]:
START_TRAINING = False

if START_TRAINING:
    for fold in tqdm(FOLDS, desc="학습 회차", unit="fold"):
        checkpoints = train_fold(fold)
        print(f"Fold {fold} · {run_dir(fold) / 'weights'}")
        print([path.name for path in checkpoints.values()], "best.pt", "last.pt")
else:
    print("학습하려면 START_TRAINING을 True로 바꾸세요.")

## 4. 같은 영상으로 두 에폭 채점하기

선택한 회차가 완료되면 `RUN_EVALUATION = True`로 바꿔 실행하세요. 각 원본을 첫 장부터 같은 조건(입력 640, CPU)으로 읽습니다. 직접 검출은 신뢰도 0.10 이상인 최고 점수 후보를 고르고, 정답 박스와 IoU 0.50 이상이면 정탐입니다. 시간 연결을 적용한 결과도 별도로 저장합니다.

미탐은 박스가 없거나 선택한 박스가 정답과 충분히 겹치지 않은 양성 장면입니다. 허브 세로 오차는 두 에폭이 **모두 정탐한 같은 장면**에서만 구합니다. 예측 박스 중심 y에서 수동 허브 y를 빼므로 양수는 예측이 더 아래에 있다는 뜻입니다.

In [ ]:
RUN_EVALUATION = False

if RUN_EVALUATION:
    pairs = []
    backgrounds = []
    for fold in tqdm(FOLDS, desc="평가 회차", unit="fold"):
        data, split_csv = inputs[fold]
        pair, background = evaluate_fold(fold, data, split_csv)
        pairs.append(pair)
        backgrounds.append(background)
    frames = pd.concat(pairs, ignore_index=True)
    background = pd.concat(backgrounds, ignore_index=True)
    summary = summarize_folds(frames)
    # 일부 회차만 돌린 결과와 전체 결과는 서로 다른 폴더에 보관합니다.
    tag = "_".join(str(fold) for fold in sorted(FOLDS))
    report = OUTPUT / f"comparison_{tag}"
    report.mkdir(parents=True, exist_ok=True)
    summary.to_csv(report / "summary.csv", index=False)
    frames.to_csv(report / "frames.csv", index=False)
    background.to_csv(report / "background.csv", index=False)
    print("비교 결과", report)
else:
    print("학습 완료 후 RUN_EVALUATION을 True로 바꾸세요.")

## 5. 미탐 수와 세로 오차를 함께 읽기

첫 표에서 동작 중 미탐과 공통 정탐 수를 먼저 보고, 같은 장면의 허브 세로 평균 절대 오차(px)를 비교하세요. ‘선택 회차 합계’는 모든 대상 프레임을 한 번씩 모아 계산합니다. 그래프는 양성 전체의 직접 검출 기준입니다. 낮을수록 좋으며 공통 정탐이 없으면 오차는 빈칸입니다. 두 번째 표에는 95에폭에서 복구하거나 잃은 프레임이 나옵니다.

In [ ]:
if RUN_EVALUATION:
    direct = summary[summary["방식"].eq("직접 검출")]
    display(direct[direct["영상"].eq("전체")].round(2))
    changes = frames[frames.change.isin(["복구", "손실"]) & frames.method.eq("YOLO")]
    display(changes[["fold", "video_id", "frame_index", "in_motion", "change", "iou_65", "iou_95"]])
    totals = direct[direct["범위"].eq("양성 전체") & direct["영상"].eq("전체")
                    & direct["fold"].isin(FOLDS)]
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
    for ax, column, title in zip(axes, ["미탐", "허브 세로 평균 절대 오차(px)"],
                                 ["Missed frames", "Vertical hub MAE (px)"]):
        table = totals.pivot(index="fold", columns="에폭", values=column)
        table.plot.bar(ax=ax, rot=0)
        ax.set(xlabel="Held-out fold", ylabel=title)
        ax.legend(title="Epoch")
    fig.tight_layout()
    fig.savefig(report / "fold_metrics.png", dpi=150)
    plt.show()

## 6. 선수별 결과와 배경 오탐 확인

선수별로 미탐 감소와 세로 오차 감소가 함께 나타나는지 확인하세요. b05·b06에서는 원판이 없는 영상 전체에 대해 오탐이 난 프레임 비율과 최장 연속 길이를 셉니다. Fold 3은 배경 전용 선수가 없어 해당 비율이 빈칸입니다.

In [ ]:
if RUN_EVALUATION:
    display(summary[summary["영상"].ne("전체") & summary["범위"].eq("양성 전체")].round(2))
    display(background.round(3))